<a href="https://colab.research.google.com/github/njwbilll/Tugas-5_Grokking-Deep-Learning-MANNING_Najwa-Bilqis-Al-Khalidah/blob/main/12_Recurrent_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 12: Jaringan Saraf yang Menulis layaknya Shakespeare

**Referensi:** Grokking Deep Learning, Andrew W. Trask (Manning Publications, 2019)

## Ringkasan Bab

Bab kedua belas memperkenalkan terobosan monumental untuk menangani data yang mengalir seiring berjalannya waktu. Arsitektur masa lalu hanya mampu menerima input dengan ukuran statis. Namun di dunia nyata, untaian kalimat lisan maupun tulisan memiliki panjang yang dinamis dan tak terduga. Untuk menyelesaikan masalah ini, buku mendemonstrasikan penciptaan Lapisan Berulang atau Recurrent Neural Networks. Arsitektur ini memiliki matriks transisi khusus yang memungkinkannya mengingat konteks kata demi kata secara berurutan, sehingga jaringan ini bukan hanya bisa membaca, melainkan bisa menciptakan kalimat baru layaknya penulis sungguhan.

## Pemahaman Teori Memori Dinamis dan Matriks Transisi

### A. Keterbatasan Komputasi Panjang Absolut
Sebuah kalimat bisa berisi tiga kosakata, sementara kalimat lainnya bisa berisi puluhan kosakata. Membangun model statis untuk setiap kemungkinan panjang kalimat sangatlah mustahil. Jaringan Saraf Berulang mengatasi hal ini dengan membaca kalimat satu per satu, mengalirkan informasi dari masa lalu untuk digabungkan dengan informasi masa kini.

### B. Mekanisme Matriks Transisi
Kunci keajaiban arsitektur ini terletak pada matriks parameter tersembunyi yang menghubungkan kondisi pikiran model di waktu sebelumnya dengan kondisi pikiran di waktu saat ini. Matriks transisi ini berfungsi sebagai brankas ingatan jangka pendek. Setiap kali karakter baru masuk, matriks ini memperbarui brankas ingatan agar selaras dengan susunan narasi.

### C. Propagasi Mundur Melintasi Ruang Waktu
Karena prediksi saat ini dipengaruhi oleh seluruh karakter yang masuk sebelumnya, algoritma pembelajaran juga harus menelusuri jejak kesalahannya menembus waktu ke arah belakang. Proses kalkulasi turunan ini mengumpulkan seluruh gradien koreksi dari ujung masa depan menuju pangkal masa lalu secara berkesinambungan.

### Tahap Persiapan Pustaka Utilitas
Kita memuat modul operasional aljabar untuk mengatur pembentukan susunan matriks memori dinamis berkelanjutan.

In [1]:
import numpy as np
print("Modul aljabar waktu dinamis bersiaga mengeksekusi komputasi memori transisi")

Modul aljabar waktu dinamis bersiaga mengeksekusi komputasi memori transisi


### Konstruksi Ekosistem Data Bahasa Sekuensial
Sebagai materi simulasi, kita memahat sebuah penggalan frasa singkat. Algoritma akan menelan frasa ini dan mencoba menebak karakter selanjutnya berdasarkan sejarah karakter yang mendahuluinya.

In [2]:
teks_korpus = "cerita kecerdasan buatan"
kumpulan_karakter = list(set(teks_korpus))
kamus_karakter = {}
indeks_angka = 0

for entitas in kumpulan_karakter:
    kamus_karakter[entitas] = indeks_angka
    indeks_angka = np.add(indeks_angka, 1)

jumlah_kosakata = len(kumpulan_karakter)
print("Total perbendaharaan huruf unik yang berhasil diregistrasi:", jumlah_kosakata)

Total perbendaharaan huruf unik yang berhasil diregistrasi: 13


### Inisialisasi Tiga Komponen Parameter Ingatan
Jaringan Saraf Berulang membutuhkan tiga blok matriks utama meliputi penyaring gerbang masuk, penjaga transisi memori antar waktu, serta pembuat keputusan akhir di pintu keluar.

In [3]:
np.random.seed(1)
dimensi_memori = 15

acak_input = np.multiply(np.random.rand(jumlah_kosakata, dimensi_memori), 0.2)
bobot_gerbang_masuk = np.subtract(acak_input, 0.1)

acak_transisi = np.multiply(np.random.rand(dimensi_memori, dimensi_memori), 0.2)
bobot_memori_transisi = np.subtract(acak_transisi, 0.1)

acak_output = np.multiply(np.random.rand(dimensi_memori, jumlah_kosakata), 0.2)
bobot_gerbang_keluar = np.subtract(acak_output, 0.1)

print("Ketiga pilar jembatan waktu memori sukses diinisialisasi")

Ketiga pilar jembatan waktu memori sukses diinisialisasi


### Deklarasi Kurva Probabilitas Serta Transformasi Sigmoid
Demi memastikan pembebasan mutlak dari penggunaan lambang pengurangan, rumus inversi dikonstruksi memanfaatkan fungsi pengurangan presisi bawaan dari pustaka numerik.

In [4]:
def aktivasi_sigmoid(variabel_x):
    inversi_tanda = np.subtract(0.0, variabel_x)
    penyebut_fraksi = np.add(1.0, np.exp(inversi_tanda))
    return np.divide(1.0, penyebut_fraksi)

def turunan_sigmoid(hasil_sigmoid):
    selisih_satu = np.subtract(1.0, hasil_sigmoid)
    return np.multiply(hasil_sigmoid, selisih_satu)

def kalkulasi_softmax(vektor_mentah):
    puncak_vektor = np.max(vektor_mentah, axis=1, keepdims=True)
    vektor_stabil = np.subtract(vektor_mentah, puncak_vektor)
    kekuatan_eksponensial = np.exp(vektor_stabil)
    akumulasi_pembagi = np.sum(kekuatan_eksponensial, axis=1, keepdims=True)
    return np.divide(kekuatan_eksponensial, akumulasi_pembagi)

print("Gudang perangkat fungsi matematika nonlinear bersiaga")

Gudang perangkat fungsi matematika nonlinear bersiaga


### Pemetaan Teks Fisik Menjadi Barisan Identitas Posisi
Mengonversi frasa cerita ke dalam susunan matriks sekuensial yang bisa dicerna satu persatu menembus dimensi waktu oleh algoritma kecerdasan.

In [5]:
def konversi_teks_ke_matriks(untaian_teks):
    matriks_representasi = np.zeros((len(untaian_teks), jumlah_kosakata))
    indeks_langkah = 0
    for alfabet in untaian_teks:
        kode_unik = kamus_karakter[alfabet]
        matriks_representasi[indeks_langkah, kode_unik] = 1.0
        indeks_langkah = np.add(indeks_langkah, 1)
    return matriks_representasi

matriks_cerita_lengkap = konversi_teks_ke_matriks(teks_korpus)
print("Format dimensi narasi utuh:", matriks_cerita_lengkap.shape)

Format dimensi narasi utuh: (24, 13)


### Simulasi Akbar Eksekusi Jaringan Saraf Waktu Dinamis
Di puncak materi bab dua belas, model dioperasikan membaca setiap huruf untuk memprediksi huruf selanjutnya secara berkesinambungan. Seusai menyelesaikan frasa di babak maju, gelombang koreksi disuntikkan menyusuri gerbang waktu arah berlawanan untuk mempertajam naluri penyusunan memori jangka pendek.

In [6]:
laju_adaptasi_bahasa = 0.1
batas_aliran_waktu = np.subtract(len(teks_korpus), 1)
batas_logaritma = 0.00000001

for putaran_epos in range(200):
    agregasi_kerugian = 0.0

    gudang_ingatan = [np.zeros((1, dimensi_memori))]
    gudang_prediksi = []
    gudang_masukan = []

    for rentang_waktu in range(batas_aliran_waktu):
        fakta_saat_ini = matriks_cerita_lengkap[rentang_waktu:np.add(rentang_waktu, 1)]
        fakta_sasaran = matriks_cerita_lengkap[np.add(rentang_waktu, 1):np.add(rentang_waktu, 2)]

        ingatan_periode_lalu = gudang_ingatan[rentang_waktu]

        dorongan_baru = np.dot(fakta_saat_ini, bobot_gerbang_masuk)
        dorongan_masa_lalu = np.dot(ingatan_periode_lalu, bobot_memori_transisi)

        kombinasi_wawasan = np.add(dorongan_baru, dorongan_masa_lalu)
        ingatan_periode_kini = aktivasi_sigmoid(kombinasi_wawasan)

        gudang_ingatan.append(ingatan_periode_kini)
        gudang_masukan.append(fakta_saat_ini)

        kesimpulan_mentah = np.dot(ingatan_periode_kini, bobot_gerbang_keluar)
        kesimpulan_probabilitas = kalkulasi_softmax(kesimpulan_mentah)
        gudang_prediksi.append(kesimpulan_probabilitas)

        komponen_logaritma = np.log(np.add(kesimpulan_probabilitas, batas_logaritma))
        kalkulasi_lintas_entropi = np.multiply(fakta_sasaran, komponen_logaritma)
        agregasi_kerugian = np.add(agregasi_kerugian, np.subtract(0.0, np.sum(kalkulasi_lintas_entropi)))

    gradien_gerbang_masuk = np.zeros_like(bobot_gerbang_masuk)
    gradien_memori_transisi = np.zeros_like(bobot_memori_transisi)
    gradien_gerbang_keluar = np.zeros_like(bobot_gerbang_keluar)

    akumulasi_gradien_masa_depan = np.zeros((1, dimensi_memori))
    urutan_mundur = reversed(range(batas_aliran_waktu))

    for arus_balik in urutan_mundur:
        probabilitas_relevan = gudang_prediksi[arus_balik]
        sasaran_relevan = matriks_cerita_lengkap[np.add(arus_balik, 1):np.add(arus_balik, 2)]
        masukan_relevan = gudang_masukan[arus_balik]

        ingatan_saat_itu = gudang_ingatan[np.add(arus_balik, 1)]
        ingatan_pemicu = gudang_ingatan[arus_balik]

        selisih_fatal = np.subtract(probabilitas_relevan, sasaran_relevan)
        gradien_gerbang_keluar = np.add(gradien_gerbang_keluar, np.dot(ingatan_saat_itu.T, selisih_fatal))

        gradien_pos_internal = np.add(np.dot(selisih_fatal, bobot_gerbang_keluar.T), akumulasi_gradien_masa_depan)
        revisi_murni_internal = np.multiply(gradien_pos_internal, turunan_sigmoid(ingatan_saat_itu))

        gradien_gerbang_masuk = np.add(gradien_gerbang_masuk, np.dot(masukan_relevan.T, revisi_murni_internal))
        gradien_memori_transisi = np.add(gradien_memori_transisi, np.dot(ingatan_pemicu.T, revisi_murni_internal))

        akumulasi_gradien_masa_depan = np.dot(revisi_murni_internal, bobot_memori_transisi.T)

    pembaruan_masuk = np.multiply(gradien_gerbang_masuk, laju_adaptasi_bahasa)
    bobot_gerbang_masuk = np.subtract(bobot_gerbang_masuk, pembaruan_masuk)

    pembaruan_transisi = np.multiply(gradien_memori_transisi, laju_adaptasi_bahasa)
    bobot_memori_transisi = np.subtract(bobot_memori_transisi, pembaruan_transisi)

    pembaruan_keluar = np.multiply(gradien_gerbang_keluar, laju_adaptasi_bahasa)
    bobot_gerbang_keluar = np.subtract(bobot_gerbang_keluar, pembaruan_keluar)

    if (np.add(putaran_epos, 1) % 40) == 0:
        status_sekuensial = np.add(putaran_epos, 1)
        print("Pencapaian lintasan waktu putaran", status_sekuensial, "mendapati rekor error komulatif", agregasi_kerugian)

print("\nJaringan berhasil menemukan harmoni relasi penyusunan linguistik")

Pencapaian lintasan waktu putaran 40 mendapati rekor error komulatif 46.70194243135839
Pencapaian lintasan waktu putaran 80 mendapati rekor error komulatif 22.90406733069192
Pencapaian lintasan waktu putaran 120 mendapati rekor error komulatif 13.633921120281723
Pencapaian lintasan waktu putaran 160 mendapati rekor error komulatif 9.169820088822954
Pencapaian lintasan waktu putaran 200 mendapati rekor error komulatif 4.527798855584361

Jaringan berhasil menemukan harmoni relasi penyusunan linguistik


### Demonstrasi Proses Berkarya Mencetak Teks Otomatis
Sesudah sistem menelan hikmah dari seluruh siklus propagasi mundur, kita memberikan satu huruf awalan acak kemudian membiarkan mesin berkhayal menciptakan rangkaian tulisan secara otonom melampaui batas batas instruksi utamanya.

In [7]:
karangan_kreatif = "c"
memori_kreativitas = np.zeros((1, dimensi_memori))

for urutan_generasi in range(25):
    huruf_terakhir = karangan_kreatif[np.subtract(len(karangan_kreatif), 1)]
    vektor_katalis = konversi_teks_ke_matriks(huruf_terakhir)

    dorongan_input = np.dot(vektor_katalis, bobot_gerbang_masuk)
    dorongan_ingatan = np.dot(memori_kreativitas, bobot_memori_transisi)

    memori_kreativitas = aktivasi_sigmoid(np.add(dorongan_input, dorongan_ingatan))
    probabilitas_huruf_baru = kalkulasi_softmax(np.dot(memori_kreativitas, bobot_gerbang_keluar))

    indeks_terbesar = np.argmax(probabilitas_huruf_baru)

    huruf_pemenang = ""
    for kunci_teks, nilai_angka in kamus_karakter.items():
        if nilai_angka == indeks_terbesar:
            huruf_pemenang = kunci_teks

    karangan_kreatif = karangan_kreatif + huruf_pemenang

print("Hasil karya mesin kecerdasan buatan menyusun literatur secara fasih:")
print(karangan_kreatif)

Hasil karya mesin kecerdasan buatan menyusun literatur secara fasih:
cerita kecerdasan buatan b
